# VQ-VAE-2 Latent Feature Extraction

Extract and visualize latent representations from all 3 hierarchical levels of the VQ-VAE-2 encoder.

The new model architecture includes a `content_proj` layer (Conv3d) that projects content-only channels
back to `hidden_channels` after the level-0 encoder, so deeper encoders receive a full-channel spatial map.
This notebook correctly accounts for that by:
- Building the model with `content_size` and `style_size` (required to reconstruct `content_proj`)
- Using `pool_only=True` to get pooled `(B, C)` vectors per level (memory-efficient)
- Separately extracting **content** vs **style** channel subsets at level 0 using the Gumbel mask
- Visualising T1 vs T2 separation and diagnosis clustering at each hierarchical level

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import json
import os

# Import local modules
import vqvae
import utils
from utils import load_data, CreateBrainMaskd, ApplyBrainMaskd

# ============================================================
# CONFIGURATION
# ============================================================
CHECKPOINT_PATH = "/home/ng24/projects/multiview-crl/results/ADNI_registered/vqvae-normal-logging-4/vqvae_model.pt"
DATA_DIR = "/data/natalia/ADNI_registered"
CSV_PATH = "/home/ng24/projects/nmpevqvae/labels_cleaned_3class.csv"
NUM_SAMPLES = 100  # number of subjects to process
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ADNI 2-view split (must match training)
CONTENT_SIZE = 256  # dimensions treated as content (shared between T1/T2)
STYLE_SIZE = 256  # dimensions treated as style (view-specific)

# Resampling: "original" (1 mm, ~182x218x182) or "2mm" (91x109x91)
RESAMPLE_MODE = "original"
# ============================================================

print(f"Using device: {DEVICE}")

## 1. Load checkpoint & build model

In [ ]:
# Load settings saved alongside the checkpoint
settings_path = os.path.join(os.path.dirname(CHECKPOINT_PATH), "settings.json")
with open(settings_path, "r") as f:
    settings = json.load(f)

print("Model settings:")
for k in [
    "vqvae_hidden_channels",
    "vqvae_res_channels",
    "vqvae_nb_res_layers",
    "vqvae_nb_levels",
    "vqvae_embed_dim",
    "vqvae_nb_entries",
    "vqvae_scaling_rates",
]:
    print(f"  {k}: {settings.get(k, 'N/A')}")

In [ ]:
# Build model — content_size / style_size are required to reconstruct content_proj
vqvae_model = vqvae.VQVAE(
    in_channels=1,
    hidden_channels=settings["vqvae_hidden_channels"],
    res_channels=settings["vqvae_res_channels"],
    nb_res_layers=settings.get("vqvae_nb_res_layers", 2),
    nb_levels=settings["vqvae_nb_levels"],
    embed_dim=settings["vqvae_embed_dim"],
    nb_entries=settings["vqvae_nb_entries"],
    scaling_rates=settings["vqvae_scaling_rates"],
    content_size=CONTENT_SIZE,
    style_size=STYLE_SIZE,
)

hidden_channels = settings["vqvae_hidden_channels"]
content_channels = vqvae_model.content_channels  # actual channel count (may differ due to rounding)
print(f"hidden_channels : {hidden_channels}")
print(f"content_channels: {content_channels}  (level-0 pooled vectors will have this many dims)")

# Wrap in DataParallel to match the saved checkpoint structure
vqvae_model = torch.nn.DataParallel(vqvae_model, device_ids=[0])

# Load checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
state_dict = checkpoint["encoders"]

# Handle old checkpoints that used ".blocks." instead of ".stack."
new_state_dict = {k.replace(".blocks.", ".stack."): v for k, v in state_dict.items()}

missing, unexpected = vqvae_model.load_state_dict(new_state_dict, strict=False)
if missing:
    print(f"⚠ Missing keys : {missing}")
if unexpected:
    print(f"⚠ Unexpected keys: {unexpected}")
if not missing and not unexpected:
    print("✓ Checkpoint loaded cleanly")
elif not missing:
    print("✓ Checkpoint loaded (unexpected keys are harmless extras in the file)")

vqvae_model.to(DEVICE)
vqvae_model.eval()
print(f"✓ Model loaded from step {checkpoint['step']}")
print(f"  Total params: {sum(p.numel() for p in vqvae_model.parameters()):,}")

## 2. Load dataset & transforms

In [ ]:
df = pd.read_csv(CSV_PATH)
label_values = sorted(df["Group"].unique())
label_map = {v: i for i, v in enumerate(label_values)}
label_names = {i: v for v, i in label_map.items()}
print(f"Labels: {label_map}")

items, missing = load_data(df, DATA_DIR, label_map)
print(f"Loaded {len(items)} samples, {len(missing)} missing files")
items = items[:NUM_SAMPLES]
print(f"Using first {len(items)} subjects")

In [ ]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Spacingd,
    Orientationd,
    ResizeWithPadOrCropd,
    NormalizeIntensityd,
    ToTensord,
)

base_keys = ["image_t1", "image_t2"]
mask_keys = ["mask_t1", "mask_t2"]
all_keys = base_keys + mask_keys

common_pre = [
    LoadImaged(keys=base_keys),
    EnsureChannelFirstd(keys=base_keys, channel_dim="no_channel"),
    CreateBrainMaskd(keys=base_keys, mask_keys=mask_keys),
    Orientationd(keys=all_keys, axcodes="RAS"),
]

if RESAMPLE_MODE == "2mm":
    spatial_size = (91, 109, 91)
    common_pre += [
        Spacingd(keys=base_keys, pixdim=(2.0, 2.0, 2.0), mode="bilinear"),
        Spacingd(keys=mask_keys, pixdim=(2.0, 2.0, 2.0), mode="nearest"),
        ResizeWithPadOrCropd(keys=all_keys, spatial_size=spatial_size),
    ]
    print(f"Using 2 mm isotropic resampling → {spatial_size}")
elif RESAMPLE_MODE == "original":
    print("Using original 1 mm resolution (no resampling)")
else:
    raise ValueError(f"Unknown RESAMPLE_MODE: {RESAMPLE_MODE}")

val_transforms = Compose(
    common_pre
    + [
        NormalizeIntensityd(keys=base_keys, nonzero=True, channel_wise=True),
        ApplyBrainMaskd(keys=base_keys, mask_keys=mask_keys, threshold=0.5),
        ToTensord(keys=base_keys),
    ]
)

## 3. Feature extraction

We call `vqvae_model(img, pool_only=True, return_recon=False)` which returns:
- `encoder_features`: list of `(B, C)` pooled vectors, one per level
  - **Level 0**: `content_channels` dims (the content-masked, projected feature)
  - **Level 1+**: `hidden_channels` dims (full channel map pooled)
- `estimated_content_indices`: the Gumbel-selected channel indices at level 0

For visualisation we also split level-0 pooled features into **content** and **style** subsets.

In [ ]:
nb_levels = settings["vqvae_nb_levels"]

# Storage
all_features = {f"level_{i}": [] for i in range(nb_levels)}
all_labels = []
all_modalities = []
all_subjects = []

# We'll also collect the raw level-0 FULL pooled features (hidden_channels dims)
# by running the encoder manually once, so we can split content vs style.
all_full_level0 = []  # (B, hidden_channels)
all_content_indices = None  # shared across samples (mask is computed per-forward from mean logits)

print(f"Extracting latent features from {len(items)} subjects (T1 + T2 each) ...")

vqvae_module = vqvae_model.module if hasattr(vqvae_model, "module") else vqvae_model

for idx, item in enumerate(items):
    if idx % 20 == 0:
        print(f"  {idx}/{len(items)} ...")

    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for modality, key in [("T1", "image_t1"), ("T2", "image_t2")]:
        img = transformed[key].unsqueeze(0).to(DEVICE)  # (1, 1, D, H, W)

        with torch.no_grad():
            _, _, enc_features, est_content_idx, _, _ = vqvae_model(
                img,
                return_recon=False,
                pool_only=True,
            )

        # enc_features is a list of (1, C) pooled tensors
        for level_idx, pooled in enumerate(enc_features):
            all_features[f"level_{level_idx}"].append(pooled.squeeze(0).cpu().float().numpy())  # (C,)

        # For level-0: also grab the FULL hidden_channels pooled features
        # by running just encoder[0] directly (before content projection path)
        with torch.no_grad():
            spatial_l0 = vqvae_module.encoders[0](img)  # (1, hidden_channels, D, H, W)
            full_pool = spatial_l0.mean(dim=[2, 3, 4])  # (1, hidden_channels)
        all_full_level0.append(full_pool.squeeze(0).cpu().float().numpy())

        # Record the content indices from the last forward pass
        if all_content_indices is None and est_content_idx is not None:
            all_content_indices = est_content_idx[0]  # list of ints

        all_labels.append(item["label"])
        all_modalities.append(modality)
        all_subjects.append(item.get("subject", idx))

# Convert to numpy
for key in all_features:
    all_features[key] = np.array(all_features[key])
all_full_level0 = np.array(all_full_level0)
all_labels = np.array(all_labels)
all_modalities = np.array(all_modalities)

print("\n✓ Feature extraction complete!")
for key, val in all_features.items():
    print(f"  {key}: {val.shape}")
print(f"  full level-0 (hidden_channels): {all_full_level0.shape}")

# Derive content / style subsets from the full level-0 features
all_style_indices = [i for i in range(hidden_channels) if i not in set(all_content_indices or [])]
print(f"  content indices ({len(all_content_indices)} ch): {all_content_indices[:8]}...")
print(f"  style indices   ({len(all_style_indices)} ch):   {all_style_indices[:8]}...")

## 4. Feature statistics

In [ ]:
print("=" * 65)
print("Feature Statistics by Level (pooled, before quantization)")
print("=" * 65)

t1_mask = all_modalities == "T1"
t2_mask = all_modalities == "T2"

for level_idx in range(nb_levels):
    features = all_features[f"level_{level_idx}"]
    t1_f = features[t1_mask]
    t2_f = features[t2_mask]
    paired_dist = np.linalg.norm(t1_f - t2_f, axis=1)
    print(f"\nLevel {level_idx}  (shape {features.shape}):")
    print(
        f"  mean={features.mean():.4f}  std={features.std():.4f}  "
        f"min={features.min():.4f}  max={features.max():.4f}"
    )
    print(f"  T1-T2 paired ℓ2 distance — mean: {paired_dist.mean():.4f}  std: {paired_dist.std():.4f}")

# Extra breakdown for level 0: content vs style channels
if all_content_indices is not None:
    print("\n--- Level 0 channel breakdown (full hidden_channels pool) ---")
    content_f = all_full_level0[:, all_content_indices]
    style_f = all_full_level0[:, all_style_indices] if all_style_indices else None
    ct1 = content_f[t1_mask]
    ct2 = content_f[t2_mask]
    print(
        f"  Content  ({len(all_content_indices)} ch)  — T1-T2 paired dist: {np.linalg.norm(ct1-ct2,axis=1).mean():.4f}"
    )
    if style_f is not None and len(style_f):
        st1 = style_f[t1_mask]
        st2 = style_f[t2_mask]
        print(
            f"  Style    ({len(all_style_indices)} ch)  — T1-T2 paired dist: {np.linalg.norm(st1-st2,axis=1).mean():.4f}"
        )
    print("  (Content dist should be smaller than style dist if disentanglement is working)")

## 5. PCA visualisation

In [ ]:
colors_by_label = plt.cm.tab10(np.linspace(0, 1, max(len(label_map), 3)))

fig, axes = plt.subplots(2, nb_levels, figsize=(5 * nb_levels, 9))
if nb_levels == 1:
    axes = axes[:, np.newaxis]  # keep 2-D indexing

for level_idx in range(nb_levels):
    features = all_features[f"level_{level_idx}"]
    pca = PCA(n_components=2)
    f2d = pca.fit_transform(features)
    ev = pca.explained_variance_ratio_

    # Row 0 — colour by diagnosis
    ax = axes[0, level_idx]
    for lab_idx, lab_name in label_names.items():
        m = all_labels == lab_idx
        ax.scatter(
            f2d[m, 0],
            f2d[m, 1],
            c=[colors_by_label[lab_idx]],
            label=lab_name,
            alpha=0.65,
            s=18,
        )
    ax.set_title(f"Level {level_idx} — Diagnosis\n({ev.sum()*100:.1f}% var)")
    ax.legend(fontsize=8)
    ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

    # Row 1 — colour by modality
    ax = axes[1, level_idx]
    for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
        m = all_modalities == mod
        ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
    ax.set_title(f"Level {level_idx} — Modality (T1 vs T2)")
    ax.legend(fontsize=8)
    ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

plt.suptitle("VQ-VAE-2 Encoder Features — PCA (pooled, before quantization)", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig("latent_features_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved latent_features_pca.png")

## 6. t-SNE visualisation

In [ ]:
fig, axes = plt.subplots(2, nb_levels, figsize=(5 * nb_levels, 9))
if nb_levels == 1:
    axes = axes[:, np.newaxis]

for level_idx in range(nb_levels):
    features = all_features[f"level_{level_idx}"]
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(features) - 1))
    f2d = tsne.fit_transform(features)

    ax = axes[0, level_idx]
    for lab_idx, lab_name in label_names.items():
        m = all_labels == lab_idx
        ax.scatter(
            f2d[m, 0],
            f2d[m, 1],
            c=[colors_by_label[lab_idx]],
            label=lab_name,
            alpha=0.65,
            s=18,
        )
    ax.set_title(f"Level {level_idx} — Diagnosis")
    ax.legend(fontsize=8)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

    ax = axes[1, level_idx]
    for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
        m = all_modalities == mod
        ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
    ax.set_title(f"Level {level_idx} — Modality (T1 vs T2)")
    ax.legend(fontsize=8)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

plt.suptitle(
    "VQ-VAE-2 Encoder Features — t-SNE (pooled, before quantization)",
    y=1.02,
    fontsize=13,
)
plt.tight_layout()
plt.savefig("latent_features_tsne.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved latent_features_tsne.png")

## 7. Content vs Style channel visualisation (level 0)

The Gumbel mask at level 0 selects `content_channels` out of `hidden_channels`.
If the model has learned to disentangle content (shared across T1/T2) from style
(view-specific), the **content** PCA should show T1 and T2 mixed together, while
the **style** PCA should separate them clearly.

In [ ]:
if all_content_indices is not None and len(all_style_indices) > 0:
    content_feats = all_full_level0[:, all_content_indices]
    style_feats = all_full_level0[:, all_style_indices]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    for row, (feats, title_sfx) in enumerate(
        [
            (content_feats, f"Content ({len(all_content_indices)} ch)"),
            (style_feats, f"Style   ({len(all_style_indices)} ch)"),
        ]
    ):
        pca = PCA(n_components=2)
        f2d = pca.fit_transform(feats)
        ev = pca.explained_variance_ratio_

        # Col 0 — by diagnosis
        ax = axes[row, 0]
        for lab_idx, lab_name in label_names.items():
            m = all_labels == lab_idx
            ax.scatter(
                f2d[m, 0],
                f2d[m, 1],
                c=[colors_by_label[lab_idx]],
                label=lab_name,
                alpha=0.65,
                s=18,
            )
        ax.set_title(f"Level-0 {title_sfx}\nBy Diagnosis ({ev.sum()*100:.1f}% var)")
        ax.legend(fontsize=8)
        ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

        # Col 1 — by modality
        ax = axes[row, 1]
        for mod, color in [("T1", "steelblue"), ("T2", "tomato")]:
            m = all_modalities == mod
            ax.scatter(f2d[m, 0], f2d[m, 1], c=color, label=mod, alpha=0.65, s=18)
        ax.set_title(f"Level-0 {title_sfx}\nBy Modality")
        ax.legend(fontsize=8)
        ax.set_xlabel(f"PC1 ({ev[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({ev[1]*100:.1f}%)")

    plt.suptitle("Level-0 Content vs Style Channel Subsets — PCA", y=1.02, fontsize=13)
    plt.tight_layout()
    plt.savefig("latent_content_vs_style_pca.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ Saved latent_content_vs_style_pca.png")
else:
    print("Skipped — content_proj not active or all channels are content.")

## 8. Reconstruction visualisation

Compare original slices with their VQ-VAE-2 reconstructions.

In [ ]:
N_SHOW = 3  # number of subjects to show

fig, axes = plt.subplots(N_SHOW, 4, figsize=(14, 4 * N_SHOW))
if N_SHOW == 1:
    axes = axes[np.newaxis, :]

for row, item in enumerate(items[:N_SHOW]):
    data_dict = {"image_t1": item["image"], "image_t2": item["z_image"]}
    transformed = val_transforms(data_dict)

    for col_offset, (key, mod_label) in enumerate([("image_t1", "T1"), ("image_t2", "T2")]):
        img = transformed[key].unsqueeze(0).to(DEVICE)  # (1, 1, D, H, W)

        with torch.no_grad():
            recon, _, _, _, _, _ = vqvae_model(img, return_recon=True, pool_only=False)

        # Mid-sagittal slice (dim 2 = D)
        mid = img.shape[2] // 2
        orig_slice = img[0, 0, mid].cpu().numpy()
        recon_slice = recon[0, 0, mid].cpu().float().numpy()

        axes[row, col_offset * 2].imshow(orig_slice, cmap="gray", origin="lower")
        axes[row, col_offset * 2].set_title(f"Subject {row} — {mod_label} original")
        axes[row, col_offset * 2].axis("off")

        axes[row, col_offset * 2 + 1].imshow(recon_slice, cmap="gray", origin="lower")
        axes[row, col_offset * 2 + 1].set_title(f"Subject {row} — {mod_label} reconstruction")
        axes[row, col_offset * 2 + 1].axis("off")

plt.suptitle("VQ-VAE-2 Reconstructions (mid-sagittal)", fontsize=13)
plt.tight_layout()
plt.savefig("reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved reconstructions.png")

## 9. Save extracted features

In [ ]:
save_dict = {
    "labels": all_labels,
    "modalities": all_modalities,
    "label_map_keys": np.array(list(label_map.keys())),
    "label_map_values": np.array(list(label_map.values())),
    "checkpoint_step": np.array([checkpoint["step"]]),
    "content_indices": np.array(all_content_indices) if all_content_indices else np.array([]),
    "full_level0": all_full_level0,
}
for i in range(nb_levels):
    save_dict[f"features_level_{i}"] = all_features[f"level_{i}"]

np.savez("extracted_latents.npz", **save_dict)
print("✓ Saved extracted_latents.npz")
for i in range(nb_levels):
    print(f"  level_{i}: {all_features[f'level_{i}'].shape}")
print(f"  full_level0: {all_full_level0.shape}")